In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
import joblib
import os

# 집계된 요일·시간 프로필만 있으므로, 현재 데이터만으로 미래 날짜 예측 성능을 충분히 검증할 수 없다는 점을 명시
# 현재 점수는 관광지 간 절대적인 혼잡도 비교가 아니라, 각 관광지의 평소 패턴 안에서 상대적으로 얼마나 붐비는지를 나타내는 지표로 해석
# 실제 관광지별 날씨에 따른 사용자수를 파악할 수 없으므로, 결과는 실측으로 검증된 예측치가 아니라 가정에 따른 보정값임
# 월별 L1 가중치: 3년치 월별 입장객 수로 별도 산출한 지수

# ==========================================
# 0. 원본 데이터 및 메타데이터 로드 (필수 준비)
# ==========================================

# 1) 방문객 프로필 데이터 (휴관일 체크 및 모델 학습용)
df_visitor = pd.read_csv('data/processed/visitor_profiles.csv', encoding='utf-8-sig')
df_visitor.columns = [str(col).strip() for col in df_visitor.columns]

# 2) 관광지 메타데이터 (실내/실외 구분 컬럼 포함) -> '관광지명' 컬럼명 정상 반영
df_meta = pd.read_excel('카카오맵_기준_관광지_리스트_fin.xlsx')
df_meta.columns = [str(c).strip() for c in df_meta.columns]

# 빠른 조회를 위해 '관광지명' -> '실내구분' 맵 생성
indoor_status_map = {}
if '실내구분' in df_meta.columns and '관광지명' in df_meta.columns:
    for _, row in df_meta.iterrows():
        place = str(row['관광지명']).strip()
        status = str(row['실내구분']).strip() # 예: '실내', '야외' 등
        indoor_status_map[place] = status

# 0-1. 컬럼 전처리

df_visitor['profile_day'] = (
    df_visitor['profile_day']
    .astype(str)
    .str.strip()
    .str.lower()
)

df_visitor['hour'] = pd.to_numeric(
    df_visitor['hour'],
    errors='coerce'
)

df_visitor['visitor_uv'] = pd.to_numeric(
    df_visitor['visitor_uv'],
    errors='coerce'
)

df_visitor = df_visitor.dropna(
    subset=['profile_day', 'hour', 'visitor_uv']
)

# ==========================================
# 1. CatBoost 모델 준비 (학습 또는 로드)
# ==========================================

model_filename = 'relative_congestion_score.pkl'

if os.path.exists(model_filename):
    model = joblib.load(model_filename)
    print("저장된 CatBoost 모델을 성공적으로 불러왔습니다!")
else:
    print("모델 파일이 없어 새로 학습을 시작합니다...")
    df_train_target = df_visitor[df_visitor['visitor_uv'] != -1].copy()
    min_vals = df_train_target.groupby('place_name')['visitor_uv'].transform('min')
    max_vals = df_train_target.groupby('place_name')['visitor_uv'].transform('max')
    is_same = (max_vals == min_vals)

    df_train_target['congestion_score'] = ((df_train_target['visitor_uv'] - min_vals) / (max_vals - min_vals)) * 99 + 1
    df_train_target.loc[is_same, 'congestion_score'] = 50.0

    weekend_days = ['saturday', 'sunday']
    df_train_target['profile_day_lower'] = df_train_target['profile_day'].str.lower().str.strip()
    df_train_target['is_weekend'] = df_train_target['profile_day_lower'].isin(weekend_days).astype(int)

    features = ['place_name', 'profile_day', 'hour', 'is_weekend']
    target = 'congestion_score'

    X = df_train_target[features]
    y = df_train_target[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function='RMSE',
        random_seed=42,
        verbose=0
    )

    model.fit(X_train, y_train, cat_features=['place_name', 'profile_day'])
    joblib.dump(model, model_filename)
    print("모델 학습 및 저장 완료!")


# ==========================================
# 2. L1 적용된 월별 가중치 파일 직접 로드 및 전처리
# ==========================================
WEIGHT_FILE = '관광지별_월별_최종_L1정규화.xlsx'

df_weight = pd.read_excel(WEIGHT_FILE)
df_weight.columns = [str(c).strip() for c in df_weight.columns]

def load_l1_monthly_weights(df_weight):
    month_cols = [f'{m}월' for m in range(1, 13)]
    place_weight_map = {}
    
    for idx, row in df_weight.iterrows():
        # 예시의 '관광지' 컬럼을 기준 키로 사용
        place = str(row['관광지']).strip()
        
        # 1~12월 L1 정규화된 값을 딕셔너리로 매핑
        weights = {}
        for m in range(1, 13):
            col_name = f'{m}월'
            try:
                val = float(row[col_name])
                weights[m] = 0.0 if np.isnan(val) else val
            except (ValueError, KeyError):
                weights[m] = 1/12  # 데이터가 없거나 변환 실패 시 균등 분할
        place_weight_map[place] = weights 
    return place_weight_map

place_monthly_weights = load_l1_monthly_weights(df_weight)


# ==========================================
# 3. 실내/실외 맞춤형 세분화 날씨 보정 함수
# (기준: 야외 + 맑음 = 1.0)
# ==========================================

def get_weather_multiplier_detailed(is_indoor, weather_condition, precipitation):
    indoor_flag = '실내' in str(is_indoor)

    # 1. 비 / 눈 / 강수 상황
    if (precipitation > 0) or (weather_condition in ['비', '눈', '비/눈']):
        if indoor_flag:
            return 1.20  # 실내 + 비/눈: 맑은 야외 대비 20% 증가
        else:
            return 0.75  # 야외 + 비/눈: 맑은 야외 대비 25% 감소

    # 2. 폭염 / 한파 극한 날씨 상황
    elif weather_condition in ['폭염', '한파']:
        if indoor_flag:
            return 1.10  # 실내 + 폭염/한파: 10% 증가
        else:
            return 0.85  # 야외 + 폭염/한파: 15% 감소

    # 3. 맑음 및 그 외 평범한 날씨 (기준점)
    else:
        if indoor_flag:
            return 0.90  # 실내 + 맑음: 야외로 인파가 빠져나가 10% 소폭 감소 (-10%)
        else:
            return 1.00  # 야외 + 맑음: 절대 기준점 (1.0)

# ==========================================
# 4. 혼잡도 등급 및 색상 판정 헬퍼 함수
# ==========================================

def get_congestion_level(score):
    """
    점수에 따른 혼잡도 등급, 색상, 한글 단계를 판정하여 반환
    - 여유 (0~39): 녹색
    - 보통 (40~69): 노랑
    - 주의 (70~84): 주황
    - 혼잡 (85~100): 빨강
    """

    if score <= 39:
        return {"level": "여유", "color": "녹색", "color_code": "green"}
    elif score <= 69:
        return {"level": "보통", "color": "노랑", "color_code": "yellow"}
    elif score <= 84:
        return {"level": "주의", "color": "주황", "color_code": "orange"}
    else:
        return {"level": "혼잡", "color": "빨강", "color_code": "red"}


# ==========================================
# 5. 최종 통합 예측 파이프라인 함수
# ==========================================

def predict_congestion_final(place_name, target_date_str, hour, weather_condition='맑음', precipitation=0.0):
    dt = datetime.strptime(target_date_str, '%Y-%m-%d')
    target_month = dt.month
    profile_day = dt.strftime('%A').lower()
    is_weekend = 1 if profile_day in ['saturday', 'sunday'] else 0
    place_clean = str(place_name).strip()

    # [1단계] 휴일 룰셋 체크 (데이터상 -1인 경우 강제 차단)
    closed_check = df_visitor[
        (df_visitor['place_name'].str.strip() == place_clean) &
        (df_visitor['profile_day'].str.lower().str.strip() == profile_day) &
        (df_visitor['hour'] == hour)
    ]

    if not closed_check.empty and (closed_check['visitor_uv'] == -1).all():
        final_score = 1.0
        congestion_info = get_congestion_level(final_score)
        return {
            'place_name': place_name,
            'date': target_date_str,
            'hour': hour,
            'status': '운영 시간 외',
            'base_score': 0.0,
            'month_multiplier': 0.0,
            'weather_multiplier': 0.0,
            'final_score': final_score,
            'congestion_level': congestion_info['level'],
            'congestion_color': congestion_info['color'],
            'color_code': congestion_info['color_code']
        }


    # [2단계] CatBoost 베이스라인 모델 예측 (시간/요일 기본 패턴)
    input_data = pd.DataFrame({
        'place_name': [place_name],
        'profile_day': [profile_day],
        'hour': [hour],
        'is_weekend': [is_weekend]
    })
    base_score = model.predict(input_data)[0]


    # [3단계] L1 월별 배율 적용 (성수기/비수기 반영)
    if place_clean not in place_monthly_weights:
        month_multiplier = 1.0
        match_status = "월별데이터 미매칭 (기본 1.0)"
    else:
        l1_ratio = place_monthly_weights[place_clean].get(target_month, 1/12)
        month_multiplier = l1_ratio * 12  
        month_multiplier = max(0.3, min(2.0, month_multiplier)) # 안전 클리핑
        match_status = "정상 매칭"

       
    # [4단계] 실내구분 및 세분화된 기상 조건 반영 날씨 보정 배율 계산
    indoor_type = indoor_status_map.get(place_clean, '야외') # 정보 없으면 기본 야외로 간주
    weather_multiplier = get_weather_multiplier_detailed(indoor_type, weather_condition, precipitation)


    # [5단계] 최종 점수 계산 (기본 점수 × 월별 배율 × 날씨 보정 배율) 및 클리핑
    final_score = base_score * month_multiplier * weather_multiplier
    final_score = max(1.0, min(100.0, final_score)) # 1 ~ 100점 스케일 고정

    # 혼잡도 등급 및 색상 판정
    congestion_info = get_congestion_level(final_score)

    return {
        'place_name': place_name,
        'place_type': indoor_type,
        'date': target_date_str,
        'hour': hour,
        'weather': f"{weather_condition} (강수: {precipitation}mm)",
        'match_status': match_status,
        'base_score': round(base_score, 2),
        'month_multiplier': round(month_multiplier, 2),
        'weather_multiplier': weather_multiplier,
        'final_score': round(final_score, 2),
        'congestion_level': congestion_info['level'],     # 여유 / 보통 / 주의 / 혼잡
        'congestion_color': congestion_info['color'],     # 녹색 / 노랑 / 주황 / 빨강
        'color_code': congestion_info['color_code']       # green / yellow / orange / red (UI 개발용)
    } 

모델 파일이 없어 새로 학습을 시작합니다...
모델 학습 및 저장 완료!


In [6]:
test_indoor_rain = predict_congestion_final('국립춘천박물관', '2026-11-01', 14, weather_condition='비', precipitation=15.0)
test_outdoor_rain = predict_congestion_final('남이섬', '2026-11-01', 14, weather_condition='비', precipitation=15.0)
print("\n--- [비 오는 날 - 실내 관광지] ---")
print(test_indoor_rain)
print("\n--- [비 오는 날 - 야외 관광지] ---")
print(test_outdoor_rain)


--- [비 오는 날 - 실내 관광지] ---
{'place_name': '국립춘천박물관', 'place_type': '실내', 'date': '2026-11-01', 'hour': 14, 'weather': '비 (강수: 15.0mm)', 'match_status': '정상 매칭', 'base_score': np.float64(70.9), 'month_multiplier': 0.68, 'weather_multiplier': 1.2, 'final_score': np.float64(57.55), 'congestion_level': '보통', 'congestion_color': '노랑', 'color_code': 'yellow'}

--- [비 오는 날 - 야외 관광지] ---
{'place_name': '남이섬', 'place_type': '실외', 'date': '2026-11-01', 'hour': 14, 'weather': '비 (강수: 15.0mm)', 'match_status': '정상 매칭', 'base_score': np.float64(72.93), 'month_multiplier': 2.0, 'weather_multiplier': 0.75, 'final_score': 100.0, 'congestion_level': '혼잡', 'congestion_color': '빨강', 'color_code': 'red'}
